# Africa & USA Railway Networks — Exploratory Data Analysis

**Domain:** Global Transportation · **Sub-theme:** Railway networks (cross-border rail infrastructure, freight and passengers)

This notebook explores the SQLite warehouse `data/railway_networks.db` produced by the pipeline
(`ingest.py → transform.py → load.py`). It uses five World Bank indicators:

| Indicator | Code |
|---|---|
| Rail lines (route-km) | `IS.RRS.TOTL.KM` |
| Rail freight (million ton-km) | `IS.RRS.GOOD.MT.K6` |
| Rail passengers (million passenger-km) | `IS.RRS.PASG.KM` |
| GDP per capita (current US$) | `NY.GDP.PCAP.CD` |
| LPI infrastructure score (1–5) | `LP.LPI.INFR.XQ` |

**Questions**
1. Which African countries operate the largest rail networks, and how intensively are they used?
2. How have rail freight and passenger volumes moved over the last decade (including the 2020 shock)?
3. Does a bigger rail network go hand in hand with better perceived trade infrastructure (LPI)?
4. How large is the gap between African rail systems and the USA benchmark?

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Locate the warehouse whether the notebook is run from notebooks/ or the project root.
DB_PATH = next(p for p in [Path("../data/railway_networks.db"), Path("data/railway_networks.db")] if p.exists())
conn = sqlite3.connect(DB_PATH)

profiles = pd.read_sql("SELECT * FROM country_profiles", conn)
infra = pd.read_sql("SELECT * FROM rail_infrastructure", conn)
freight = pd.read_sql("SELECT * FROM rail_freight", conn)
passengers = pd.read_sql("SELECT * FROM rail_passengers", conn)
master = pd.read_sql("SELECT * FROM v_rail_master", conn)
names = profiles.set_index("country_code")["country_name"]

HAS_USA = "US" in set(profiles["country_code"])
print(f"Warehouse: {DB_PATH}")
for name, df in [("country_profiles", profiles), ("rail_infrastructure", infra),
                 ("rail_freight", freight), ("rail_passengers", passengers)]:
    print(f"  {name:<20} {len(df):>5} rows")
print(f"USA benchmark present in extract: {HAS_USA}")

## 1. Data coverage

Rail statistics are reported far less consistently than GDP or LPI. Before drawing conclusions we check how
many countries in the extract actually report each rail metric.

In [ ]:
coverage = pd.DataFrame({
    "countries_in_extract": [profiles.shape[0]],
    "report_rail_lines": [profiles["latest_rail_lines_km"].notna().sum()],
    "report_rail_freight": [profiles["latest_rail_freight_million_tkm"].notna().sum()],
    "report_rail_passengers": [profiles["latest_rail_passengers_million_km"].notna().sum()],
    "report_lpi_infra": [profiles["latest_lpi_infrastructure_score"].notna().sum()],
    "report_gdp_pc": [profiles["latest_gdp_per_capita_usd"].notna().sum()],
}).T.rename(columns={0: "countries"})
coverage

## 2. Largest African rail networks (Visualization 1)

Bar chart of the top 15 African countries by latest reported rail route-km (fewer bars appear when fewer
countries report the indicator).

In [ ]:
top_africa = (profiles[(profiles["is_africa"] == 1) & profiles["latest_rail_lines_km"].notna()]
              .nlargest(15, "latest_rail_lines_km"))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_africa["country_name"], top_africa["latest_rail_lines_km"],
               color=sns.color_palette("crest", len(top_africa)))
ax.invert_yaxis()
for bar, year in zip(bars, top_africa["latest_rail_lines_year"]):
    ax.text(bar.get_width() + 40, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():,.0f} km ({int(year)})", va="center", fontsize=9)
ax.set_title("Top African countries by rail network size (latest route-km)")
ax.set_xlabel("Rail lines (route-km)")
ax.set_xlim(0, top_africa["latest_rail_lines_km"].max() * 1.3)
plt.tight_layout()
plt.show()

**Insight.** Algeria and the Democratic Republic of the Congo dominate the reporting set, each with
roughly 3,600–4,000 route-km. Their networks are a legacy of colonial-era mineral and port lines (the DRC's
Katanga copper belt lines, Algeria's coastal and phosphate routes). Most other reporting networks — Botswana,
Cameroon, Congo Rep., Côte d'Ivoire and Burkina Faso — are under 1,000 km. Côte d'Ivoire and Burkina Faso
share one cross-border line, the Abidjan–Ouagadougou railway.

## 3. Rail freight trends (Visualization 2)

Freight ton-km over time for every country reporting the indicator.

In [ ]:
fr = freight.assign(country=freight["country_code"].map(names))
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=fr, x="year", y="rail_freight_million_tkm", hue="country", marker="o", ax=ax)
ax.set_title("Rail freight transported (million ton-km)")
ax.set_ylabel("Million ton-km")
ax.set_xlabel("Year")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

fr.groupby("country").agg(first_year=("year", "min"), last_year=("year", "max"),
                          mean_mtkm=("rail_freight_million_tkm", "mean"),
                          max_mtkm=("rail_freight_million_tkm", "max")).round(1)

**Insight.** Algeria and Cameroon each move about 0.8–1.0 billion ton-km of freight a year. The DRC moves
under 200 million ton-km even though its network is about four times longer than Cameroon's. Volumes
change from year to year without a clear upward trend. African rail freight has stayed flat for the past
decade while freight on most corridors has shifted to roads.

## 4. Rail network size vs. LPI infrastructure score (Visualization 3)

Does a larger rail network coincide with better-rated trade and transport infrastructure?

In [ ]:
sc = profiles.dropna(subset=["latest_rail_lines_km", "latest_lpi_infrastructure_score"])
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.scatterplot(data=sc, x="latest_rail_lines_km", y="latest_lpi_infrastructure_score",
                hue="subregion", size="latest_gdp_per_capita_usd", sizes=(60, 400), ax=ax)
for _, r in sc.iterrows():
    ax.annotate(r["country_name"], (r["latest_rail_lines_km"], r["latest_lpi_infrastructure_score"]),
                xytext=(6, 4), textcoords="offset points", fontsize=8)
if len(sc) >= 3:
    r_val = np.corrcoef(sc["latest_rail_lines_km"], sc["latest_lpi_infrastructure_score"])[0, 1]
    ax.set_title(f"Rail route-km vs LPI infrastructure score (Pearson r = {r_val:.2f}, n = {len(sc)})")
ax.set_xlabel("Rail lines (route-km, latest)")
ax.set_ylabel("LPI infrastructure score (1–5, latest)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

profiles.groupby("has_rail_network")["latest_lpi_infrastructure_score"].agg(["count", "mean"]).round(3)

**Insight.** Network length does not predict LPI infrastructure quality. Across the seven reporting
countries the correlation is weak and slightly negative. Botswana (886 km) and Côte d'Ivoire (639 km) score
highest, while the much longer Algerian and DRC networks score about 2.1–2.3. Countries that report rail
lines score only slightly higher on average than those that do not. Track condition, port integration and
operations seem to matter more than how many kilometres of track a country has.

## 5. Rail metrics heatmap by country (Visualization 4)

Latest values of each rail and context metric, min-max normalised per column so the very different units can
share one colour scale (annotations show the raw values).

In [ ]:
metric_cols = {
    "latest_rail_lines_km": "Rail km",
    "latest_rail_freight_million_tkm": "Freight Mtkm",
    "latest_rail_passengers_million_km": "Passenger Mpkm",
    "latest_freight_density_mtkm_per_km": "Freight density",
    "latest_lpi_infrastructure_score": "LPI infra",
    "latest_gdp_per_capita_usd": "GDP per capita",
}
hm = (profiles[profiles["has_rail_network"] == 1]
      .set_index("country_name")[list(metric_cols)].rename(columns=metric_cols))
normed = (hm - hm.min()) / (hm.max() - hm.min()).replace(0, np.nan)

fig, ax = plt.subplots(figsize=(10, 0.6 * len(hm) + 2))
sns.heatmap(normed, annot=hm.round(2), fmt="g", cmap="YlGnBu", linewidths=0.5,
            cbar_kws={"label": "Min-max normalised (per metric)"}, ax=ax)
ax.set_title("Rail network profile of reporting countries (latest values)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

**Insight.** Cameroon has the busiest network in the extract: about 1.0 million ton-km of freight per
route-km, roughly four times Algeria's density and twenty times the DRC's. The Douala–N'Djamena corridor,
which runs through Cameroon's rail line to landlocked Chad and the Central African Republic, likely explains
this. Botswana, Côte d'Ivoire, Burkina Faso and Congo Rep. report track length but not traffic. That gap in
the data is a policy finding in itself.

## 6. Africa vs. USA comparison (Visualization 5)

Aggregated latest-year rail metrics for Africa (sum or mean of reporting countries) next to the USA benchmark.
The USA is included automatically when the ingestion stage captures `country_code = 'US'`.

In [ ]:
def region_summary(df: pd.DataFrame) -> pd.Series:
    with_freight = df["latest_rail_freight_million_tkm"].notna()
    return pd.Series({
        "Rail lines (km)": df["latest_rail_lines_km"].sum(min_count=1),
        "Freight (million ton-km)": df["latest_rail_freight_million_tkm"].sum(min_count=1),
        "Passengers (million pkm)": df["latest_rail_passengers_million_km"].sum(min_count=1),
        "Freight density (Mtkm/km)": (df.loc[with_freight, "latest_rail_freight_million_tkm"].sum()
                                      / df.loc[with_freight, "latest_rail_lines_km"].sum()
                                      if with_freight.any() else np.nan),
        "Avg LPI infra score": df["latest_lpi_infrastructure_score"].mean(),
        "Avg GDP per capita (US$)": df["latest_gdp_per_capita_usd"].mean(),
    })

comparison = pd.DataFrame({region: region_summary(grp) for region, grp in profiles.groupby("region")})
display(comparison.round(3))

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
palette = {"Africa": "#2a9d8f", "USA": "#e76f51"}
for ax, metric in zip(axes.flat, comparison.index):
    vals = comparison.loc[metric].dropna()
    ax.bar(vals.index, vals.values, color=[palette.get(r, "grey") for r in vals.index])
    ax.set_title(metric, fontsize=10)
    for i, v in enumerate(vals.values):
        ax.text(i, v, f"{v:,.2f}" if v < 100 else f"{v:,.0f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Africa vs USA — rail network and context metrics (latest values)"
             + ("" if HAS_USA else "\n(USA benchmark not present in current extract — re-run ingestion)"))
plt.tight_layout()
plt.show()

**Insight.** Taken together, the seven African networks in the extract add up to about 11,500 route-km.
That is less than a tenth of the US Class I freight system: the Association of American Railroads reports
roughly 140,000 route-miles across all US railroads, about 225,000 km. Africa's combined reported freight of
about 2 billion ton-km is a tiny share of US rail ton-km. US railroads carry roughly 1.5 trillion ton-miles a
year, which makes the US network one of the most heavily used freight systems in the world. African rail
gaps are about how heavily networks are used as well as how long they are.

*The US reference figures in this cell come from AAR public fact sheets and are given as context only. The
chart uses only data in the warehouse.*

## 7. Rail passenger trends (Visualization 6)

Grouped bar chart of passenger-km by year. The interactive Plotly version underneath lets you hover over
individual values.

In [ ]:
ps = passengers.assign(country=passengers["country_code"].map(names))
pivot = ps.pivot_table(index="year", columns="country", values="rail_passengers_million_km")

ax = pivot.plot(kind="bar", figsize=(11, 5), width=0.8, colormap="viridis")
ax.set_yscale("log")
ax.set_title("Rail passengers carried (million passenger-km, log scale)")
ax.set_ylabel("Million passenger-km (log)")
ax.set_xlabel("Year")
ax.legend(title="Country", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

import plotly.express as px
fig = px.bar(ps, x="year", y="rail_passengers_million_km", color="country", barmode="group",
             log_y=True, title="Rail passenger-km by country and year (interactive)",
             labels={"rail_passengers_million_km": "Million passenger-km", "year": "Year"})
fig.show()

**Insight.** Passenger rail took the hardest hit from COVID-19. Algeria's passenger-km fell 76% in 2020,
from about 1,454 to 348 million, and the DRC's fell about 80%. Cameroon's drop was smaller at about 11%.
Before the pandemic, Algeria's suburban and intercity services grew steadily, from 1,141 million
passenger-km in 2012 to a peak of 1,602 million in 2018. The DRC carries fewer than 30 million passenger-km
a year on a network of more than 3,600 km, which shows how run-down its passenger service is.

## 8. Key findings

1. **Rail data is sparse.** Only a handful of African countries in the extract report rail route-km, and even
   fewer report freight or passenger traffic. Better rail statistics should come before large investments.
2. **Long networks are not busy networks.** The DRC has the second-longest network but the lowest freight
   density. Cameroon has a short network and the highest density, which reflects how much Central African
   trade passes through Douala.
3. **The LPI follows quality, not quantity.** Rail route-km shows no positive correlation with the LPI
   infrastructure score.
4. **Passenger rail is fragile.** The 2020 collapse of 76–80% in Algeria and the DRC shows how exposed
   passenger services are to shocks.
5. **The USA benchmark is roughly 20 times larger.** Closing the gap needs both new track (for example the
   AfCFTA-era corridors such as the Lobito, Tanzania SGR and Abidjan–Lagos lines) and much heavier use of
   existing track.

In [ ]:
conn.close()